# cast-qml-01-cnn-embeddings

**What this does.** Castings version of weld notebook 01. For each of 25 splits (5 repeats × 5 folds, grouped by **physical part**, never by image), it fine-tunes one ResNet-18 on the training patches only (`cnnreal`, same settings as the weld study) and saves the 512-number penultimate-layer embedding of every patch plus the CNN's own test scores. It also saves two split-independent feature sets: HOG (1,764-d) and a frozen ImageNet ResNet-18 (512-d). No synthetic data is used.

**Settings:** Accelerator = GPU T4 (the notebook stops without a GPU). Internet = ON (downloads the ImageNet weights).

**Input (Add Input):** the dataset `gdxray-castings-patches` (upload `gdxray-castings-patches.zip` as a new Kaggle dataset).

**Outputs (/kaggle/working):** `cast_meta.csv`, `cast_splits.csv`, `hog_real.npy`, `frozen_real.npy`, `emb/emb_r{rep}_f{fold}.npz` (25 files), `cnn_head_perfold.csv`.

**Runtime:** about 15–30 min on a T4 (25 ResNet-18 trainings of 600 steps). Then *Save Version → Save & Run All*, and use this notebook's output as the input of cast-qml-02 and cast-qml-03.

In [ ]:
import builtins, functools, time
print = functools.partial(builtins.print, flush=True)
T_START = time.time()
def log(*a): print(f"[{time.time() - T_START:7.1f}s]", *a)
log("notebook started")

import os, socket, numpy as np, pandas as pd, cv2
from skimage.feature import hog
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, recall_score
log("libraries loaded")

def walk_find(name, root="/kaggle/input", max_depth=7):
    hits, base = [], root.rstrip("/").count("/")
    for dirpath, dirnames, filenames in os.walk(root, followlinks=True):
        if dirpath.count("/") - base >= max_depth: dirnames[:] = []; continue
        if name in filenames: hits.append(os.path.join(dirpath, name))
    return sorted(hits)

CSV = (walk_find("casting_patches.csv") or [None])[0]
if CSV is None:
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        print(dirpath, f"({len(filenames)} files)")
    raise FileNotFoundError("Add the dataset gdxray-castings-patches (Add Input)")
IMG_DIR = os.path.join(os.path.dirname(CSV), "img")
OUT = "/kaggle/working"
log("inputs found:", CSV, "|", len(os.listdir(IMG_DIR)), "patch images")

SIZE, SEED = 224, 42
REPEATS, FOLDS, VAL_FRAC = 5, 5, 0.2
STEPS, BS, LR = 600, 32, 3e-4
PRETRAINED = True
ALLOW_CPU = False

In [ ]:
def rs(c):
    interp = cv2.INTER_AREA if c.shape[0] > SIZE else cv2.INTER_LINEAR
    return cv2.resize(c, (SIZE, SIZE), interpolation=interp)

def load_real():
    meta = pd.read_csv(CSV)
    X = []
    for f in meta.file:
        A = cv2.imread(os.path.join(IMG_DIR, f), 0)
        if A is None: raise FileNotFoundError(f)
        X.append(rs(A))
    return np.array(X, np.uint8), meta.label.values.astype(int), meta.group.values, meta

def splits(groups):
    units = np.array(sorted(set(groups)))
    for rep in range(REPEATS):
        r = np.random.default_rng(SEED + rep)
        perm = r.permutation(units)
        fold_of = {u: i % FOLDS for i, u in enumerate(perm)}
        f = np.array([fold_of[x] for x in groups])
        for k in range(FOLDS):
            tr_units = sorted(set(groups[f != k]))
            val_units = set(r.choice(tr_units, max(1, int(round(VAL_FRAC * len(tr_units)))), replace=False))
            fit = np.where((f != k) & ~np.isin(groups, list(val_units)))[0]
            val = np.where(np.isin(groups, list(val_units)))[0]
            yield rep, k, fit, val, np.where(f == k)[0]

def best_threshold(yt, p):
    ts = np.unique(np.concatenate([p, [0.5]]))
    ba = [balanced_accuracy_score(yt, (p >= t).astype(int)) for t in ts]
    return float(ts[int(np.argmax(ba))])

def hog_feat(x):
    return hog(cv2.resize(x, (128, 128), interpolation=cv2.INTER_AREA) / 255.0, orientations=9,
               pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys").astype(np.float32)

In [ ]:
Xr, yr, gr, meta = load_real()
meta.insert(0, "idx", np.arange(len(meta)))
meta.to_csv(f"{OUT}/cast_meta.csv", index=False)
log(f"PATCHES: {len(yr)} | defective {yr.sum()} | clean {(yr==0).sum()} | series {meta.series.nunique()} | part groups {len(set(gr))}")
H_real = np.stack([hog_feat(x) for x in Xr])
np.save(f"{OUT}/hog_real.npy", H_real)
log("HOG features:", H_real.shape)
SPLITS = list(splits(gr))
pd.DataFrame([dict(repeat=rep, fold=k, n_fit=len(a), n_val=len(b), n_test=len(c),
                   fit_groups=";".join(sorted(set(gr[a]))), val_groups=";".join(sorted(set(gr[b]))), test_groups=";".join(sorted(set(gr[c]))))
              for rep, k, a, b, c in SPLITS]).to_csv(f"{OUT}/cast_splits.csv", index=False)
for rep, k, a, b, c in SPLITS:
    assert not (set(gr[a]) & set(gr[b])) and not (set(gr[a]) & set(gr[c])) and not (set(gr[b]) & set(gr[c]))
    assert all(len(set(yr[s])) == 2 for s in (a, b, c))
log(len(SPLITS), "splits (no part shared between fit/val/test) | example sizes fit/val/test:", [(len(a), len(b), len(c)) for _, _, a, b, c in SPLITS[:3]])

In [ ]:
import torch, torch.nn as nn, torchvision
log(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
dev = "cpu"
if torch.cuda.is_available():
    log("GPU:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
    try:
        a = torch.randn(512, 512, device="cuda"); float((a @ a).sum()); dev = "cuda"
    except Exception as e:
        log("This GPU can't run the installed PyTorch:", repr(e)[:200])
if dev != "cuda" and not ALLOW_CPU:
    raise RuntimeError("No working GPU. Settings -> Accelerator -> GPU T4 x2 (not P100, not None), then run again.")
log("device:", dev)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def prep(xb, train):
    x = torch.from_numpy(xb).to(dev).float().div(255).unsqueeze(1)
    if train:
        n = x.shape[0]
        x = torch.where(torch.rand(n, 1, 1, 1, device=dev) < 0.5, x.flip(3), x)
        x = torch.where(torch.rand(n, 1, 1, 1, device=dev) < 0.5, x.flip(2), x)
        mu = x.mean((2, 3), keepdim=True)
        x = (x - mu) * torch.empty(n, 1, 1, 1, device=dev).uniform_(0.85, 1.15) + mu
        x = x + torch.empty(n, 1, 1, 1, device=dev).uniform_(-0.06, 0.06)
        x = (x + torch.randn_like(x) * 0.01).clamp(0, 1)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

def make_model():
    w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None
    m = torchvision.models.resnet18(weights=w); m.fc = nn.Linear(512, 1)
    return m.to(dev)

def train(m, X, y, steps, lr, rng, verbose=False):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    lossf = nn.BCEWithLogitsLoss(); m.train()
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]; per = BS // 2
    for s in range(steps):
        idx = np.concatenate([rng.choice(i0, per), rng.choice(i1, per)])
        xb = prep(X[idx], True); tb = torch.from_numpy(y[idx]).float().to(dev)
        with torch.autocast(device_type="cuda", enabled=dev == "cuda"):
            loss = lossf(m(xb).squeeze(1), tb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if verbose and (s + 1) % 100 == 0: log(f"    first model: step {s + 1}/{steps}")
    return m

@torch.no_grad()
def predict_and_embed(m, X):
    m.eval(); body = nn.Sequential(*list(m.children())[:-1]); P, E = [], []
    for i in range(0, len(X), 64):
        e = body(prep(X[i:i+64], False)).flatten(1).float()
        E.append(e.cpu().numpy()); P.append(torch.sigmoid(m.fc(e)).squeeze(1).cpu().numpy())
    return np.concatenate(P), np.concatenate(E).astype(np.float16)

socket.setdefaulttimeout(120)
log("downloading ImageNet ResNet-18 weights (needs Internet ON in Settings) ...")
try:
    frozen = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
except Exception as e:
    raise RuntimeError("Could not download the ImageNet weights. Turn Internet ON (Settings -> Internet) and run again.") from e
frozen.fc = nn.Linear(512, 1); frozen = frozen.to(dev)
_, F_real = predict_and_embed(frozen, Xr)
np.save(f"{OUT}/frozen_real.npy", F_real)
log("frozen ImageNet embeddings:", F_real.shape)

In [ ]:
os.makedirs(f"{OUT}/emb", exist_ok=True)
t_loop, n_start = time.time(), len(os.listdir(f"{OUT}/emb"))
rows = pd.read_csv(f"{OUT}/cnn_head_perfold.csv").to_dict("records") if os.path.exists(f"{OUT}/cnn_head_perfold.csv") else []
for rep, k, fit, val, te in SPLITS:
    path = f"{OUT}/emb/emb_r{rep}_f{k}.npz"
    if os.path.exists(path):
        log(f"rep {rep} fold {k}: already done, skipping"); continue
    rng = np.random.default_rng(SEED + 100 * rep + k)
    torch.manual_seed(SEED + 100 * rep + k)
    t_m = time.time()
    m = train(make_model(), Xr[fit], yr[fit], STEPS, LR, rng, verbose=not rows)
    p_all, E_all = predict_and_embed(m, Xr)
    thr = best_threshold(yr[val], p_all[val]); p, yt = p_all[te], yr[te]
    yp = (p >= thr).astype(int); rec = recall_score(yt, yp, average=None, labels=[0, 1], zero_division=0)
    row = dict(repeat=rep, fold=k, model="cnn_real", val_auc=roc_auc_score(yr[val], p_all[val]),
               auc=roc_auc_score(yt, p), bal_acc=balanced_accuracy_score(yt, yp), rec_def=rec[1], rec_clean=rec[0], thr=thr)
    rows.append(row)
    log(f"rep {rep} fold {k} | AUC {row['auc']:.3f} | BA {row['bal_acc']:.3f} | model took {time.time()-t_m:.0f}s")
    if len(rows) == 1:
        took = time.time() - t_m
        log(f"    estimated total for 25 models: about {25 * took / 60:.0f} min" +
            ("  <-- far slower than a T4 (about 0.5 min per model): check that the GPU is really in use" if took > 180 else ""))
    np.savez_compressed(path, fit=fit, val=val, te=te, cnnreal_real=E_all, prob_cnnreal=p_all.astype(np.float32))
    pd.DataFrame(rows).to_csv(f"{OUT}/cnn_head_perfold.csv", index=False)
    del m; torch.cuda.empty_cache() if dev == "cuda" else None
    n_done = len(os.listdir(f"{OUT}/emb")); per = (time.time() - t_loop) / max(1, n_done - n_start)
    log(f"--- split {n_done}/{len(SPLITS)} saved | {per/60:.1f} min per split | about {per*(len(SPLITS)-n_done)/60:.0f} min left")

In [ ]:
df = pd.read_csv(f"{OUT}/cnn_head_perfold.csv")
print(df[["auc", "bal_acc", "rec_def", "rec_clean"]].agg(["mean", "std"]).round(3))
print("Files:", sorted(os.listdir(OUT)), "| emb files:", len(os.listdir(f"{OUT}/emb")))